# 01 · Build pocket–ligand graph shards

Converts raw PDB files into the compact graph dataset every other notebook consumes.

**Inputs** (attach as Kaggle Datasets)
- `protein-ligand-binding-site` — the BioLiP-style export: `nonredund_rec/`, `nonredund_lig/`,
  `PL_nonredund_annotation.csv`. Trains the **generator** (no affinity labels).
- *optional* PDBBind / PDBBind+ `refined-set` + `INDEX_*` — trains the **affinity ranker**.

**Output** → `/kaggle/working/graphs/` : `{split}_*.pt` shards + `{split}_manifest.json`,
the ligand-size prior, and a copy of `src/molgen` so later notebooks need only this one input.

**Run this with the accelerator set to *None***. It is pure CPU work, so a GPU session
wastes your GPU quota and (on P100 sessions) gives you fewer CPU cores.

### What one sample contains
A complex is stored as positions plus small integer codes — ~10 kB. Featurising it into the
dense graph the model eats costs **0.8 ms** and inflates it **58×** to ~590 kB, so the dense
form is built on the fly in DataLoader workers. Storing it instead would need **88 GB** for the
full set against Kaggle's 20 GB output limit.

In [ ]:
# --- locate the molgen package ---------------------------------------------
# On Kaggle, make `molgen` importable in one of these ways:
#   1. Add the "molgen-src" Dataset (upload this repo's `src/` folder), or
#   2. Add the output of notebook 01 (it ships a copy of `src/molgen`), or
#   3. Run the notebook from a clone of the repo.
import glob
import subprocess
import sys
from pathlib import Path


def find_molgen():
    patterns = [
        "/kaggle/input/*/src", "/kaggle/input/*", "/kaggle/input/*/*",
        "/kaggle/working/src", "./src", "../src", ".", "..",
    ]
    for pattern in patterns:
        for candidate in sorted(glob.glob(pattern)):
            if (Path(candidate) / "molgen" / "__init__.py").is_file():
                return str(Path(candidate).resolve())
    return None


MOLGEN_SRC = find_molgen()
if MOLGEN_SRC is None:
    raise FileNotFoundError(
        "Could not find the `molgen` package.\n"
        "Attach the molgen-src dataset (the repo's src/ folder, containing molgen/), "
        "or the output of notebook 01, then re-run this cell."
    )
if MOLGEN_SRC not in sys.path:
    sys.path.insert(0, MOLGEN_SRC)
print("molgen source:", MOLGEN_SRC)

In [ ]:
# --- dependencies -----------------------------------------------------------
# torch_geometric is required. We deliberately avoid torch_scatter/torch_cluster
# (they need wheels matched to the exact torch build and are the usual cause of
# Kaggle install failures) -- molgen uses scipy KD-trees and native torch ops.
# rdkit is optional: without it you lose SMILES/QED/SA but nothing else.
def ensure(module: str, package: str | None = None) -> bool:
    try:
        __import__(module)
        return True
    except ImportError:
        pkg = package or module
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    try:
        __import__(module)
        return True
    except ImportError:
        print(f"  !! {module} unavailable")
        return False


HAS_PYG = ensure("torch_geometric")
HAS_RDKIT = ensure("rdkit")
assert HAS_PYG, "torch_geometric is required"
print("rdkit:", HAS_RDKIT)

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import molgen
from molgen import (
    affinity, chem, dataset, diffusion, egnn, evaluate, graphs, pdbio, pockets, sources,
)
from molgen.train_utils import (
    EMA, History, TimeBudget, count_parameters, device_info, grad_norm,
    load_checkpoint, save_checkpoint, set_seed, warmup_cosine_lambda,
)

warnings.filterwarnings("ignore", category=UserWarning)
set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("molgen", molgen.__version__, "| device:", device)
print(device_info())

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK)

## 1 · Locate the raw data

`sources.BioLiPSource` searches `/kaggle/input` for the three pieces it needs. Pass the paths
explicitly if your dataset slug nests them differently.

In [ ]:
biolip = sources.BioLiPSource()      # or BioLiPSource(annotation_csv=..., receptor_dir=..., ligand_dir=...)
print("annotation:", biolip.annotation_csv)
print("receptors :", biolip.receptor_dir)
print("ligands   :", biolip.ligand_dir)

n_rec = sum(1 for _ in biolip.receptor_dir.glob("*.pdb"))
n_lig = sum(1 for _ in biolip.ligand_dir.glob("*.pdb"))
print(f"\n{n_rec:,} receptor files | {n_lig:,} ligand files")

## 2 · Configuration

`pocket_radius` and `max_pocket_atoms` are **baked into the shards** — changing them later
means rebuilding. The load-time cutoffs (`pocket_cutoff`, `inter_cutoff`) are not.

Set `MAX_COMPLEXES = None` for the full set. The full build is ~40 min on 4 cores; the
`TIME_BUDGET` stops cleanly and keeps whatever finished, so a partial dataset is still usable.

In [ ]:
CFG = dataset.BuildConfig(
    pocket_radius=8.0,       # A around any ligand heavy atom, whole residues kept
    max_pocket_atoms=600,    # caps memory per graph; ~600 covers a druggable site
    pocket_cutoff=4.5,       # pocket contact graph radius (load time)
    inter_cutoff=6.0,        # ligand-pocket cross edges (load time)
    center_mode="pocket",    # origin = pocket centroid, known at inference too
    min_ligand_atoms=6,
    max_ligand_atoms=48,     # must be >= the largest ligand you want to generate
    shard_size=4000,
)

MAX_COMPLEXES = None          # None = all; set e.g. 20_000 for a fast first pass
DEDUP = True                  # one entry per (assembly, ligand id)
SPLITS = (0.94, 0.03, 0.03)
TIME_BUDGET = {"train": 4.0 * 3600, "val": 900, "test": 900}

GRAPH_DIR = WORK / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
print(CFG)

## 3 · Index and filter

Rejections are counted by reason, so you can see *why* the set shrank rather than guessing.
Ions, buffers, cryoprotectants, bare amino acids and nucleotides are dropped — BioLiP calls
them ligands, but they are not generative targets.

In [ ]:
t0 = time.time()
specs = biolip.specs(dedup=DEDUP, limit=MAX_COMPLEXES)
print(f"{len(specs):,} candidate complexes indexed in {time.time() - t0:.0f}s")
print(f"{len({s.receptor_key for s in specs}):,} distinct PDB entries")

ligand_counts = pd.Series([s.ligand_name for s in specs]).value_counts()
print("\nmost common ligands:")
print(ligand_counts.head(12).to_string())

## 4 · Split by PDB entry, not at random

A random split puts `1b7z_1_OXL_D` in train and `1b7z_1_OXL_E` in test — the same protein with
the same ligand in a second copy of the site. Every metric would then be measuring memorisation.
Grouping on the 4-character PDB code is the minimum defensible unit.

In [ ]:
split_specs = sources.split_by_receptor(specs, fractions=SPLITS, seed=0)
for name, items in split_specs.items():
    print(f"{name:>5}: {len(items):>7,} complexes  {len({s.receptor_key for s in items}):>6,} PDB entries")

overlap = (
    {s.receptor_key for s in split_specs["train"]}
    & {s.receptor_key for s in split_specs["test"]}
)
assert not overlap, f"leakage: {len(overlap)} PDB entries in both train and test"
print("\nno PDB entry appears in two splits")

## 5 · Build the shards

Parsing runs in a process pool. Failures are per-complex and never abort the build.

In [ ]:
import os

manifests = {}
for name in ("val", "test", "train"):      # small splits first: fail fast
    manifests[name] = dataset.build_shards(
        split_specs[name],
        GRAPH_DIR,
        split=name,
        cfg=CFG,
        num_workers=max(1, os.cpu_count() or 2),
        chunk_size=128,
        time_budget_s=TIME_BUDGET[name],
    )
    print()

total = sum(m["num_graphs"] for m in manifests.values())
size_gb = sum(m["bytes_on_disk"] for m in manifests.values()) / 1e9
print(f"TOTAL {total:,} complexes, {size_gb:.2f} GB on disk")

In [ ]:
# why complexes were dropped, pooled over splits
rej = pd.Series(
    pd.DataFrame([m["reject_reasons"] for m in manifests.values()]).sum().sort_values(ascending=False)
)
rej.name = "count"
display(rej.to_frame().assign(pct=lambda d: (100 * d["count"] / d["count"].sum()).round(1)))

## 6 · Fit the ligand-size prior and the coordinate scale

Two numbers the generator cannot work without:

- **`SizePrior`** — a pocket-conditioned histogram of ligand size. A pocket-conditioned model
  must choose *how many atoms* to place before denoising starts; sampling that from
  `p(n_atoms | pocket residues)` keeps molecules commensurate with the cavity instead of
  always emitting the dataset mean.
- **`coord_scale`** — the std of ligand coordinates about the pocket centroid. A DDPM's
  terminal prior is unit variance, so this is the factor that maps it onto Ångström. Get it
  wrong by 2× and you either generate a tight blob at the centre or spray atoms into solvent.

In [ ]:
train_ds = dataset.ShardedComplexDataset(GRAPH_DIR, "train",
                                         pocket_cutoff=CFG.pocket_cutoff,
                                         inter_cutoff=CFG.inter_cutoff)
print(f"train dataset: {len(train_ds):,} complexes")

prior = dataset.SizePrior.fit(train_ds, max_samples=30_000, max_atoms=CFG.max_ligand_atoms)
prior.save(GRAPH_DIR / "size_prior.json")

coord_scale = diffusion.estimate_coord_scale(train_ds, max_samples=5_000)

meta = {
    "build_config": manifests["train"]["config"],
    "feature_dims": manifests["train"]["feature_dims"],
    "coord_scale": round(coord_scale, 4),
    "counts": {k: m["num_graphs"] for k, m in manifests.items()},
}
(GRAPH_DIR / "dataset_meta.json").write_text(json.dumps(meta, indent=2))

print(f"coord_scale = {coord_scale:.3f} A")
print("size prior draws for a 40-residue pocket:",
      [prior.sample(40) for _ in range(10)])
print(json.dumps(meta["feature_dims"], indent=2))

## 7 · Sanity check the dataset

Look at this before spending GPU hours. The things that go wrong here are a pocket-size
distribution pinned against `max_pocket_atoms` (raise it), an element distribution with a
surprise element (check the filter), and ligand sizes clipped at `max_ligand_atoms`.

In [ ]:
stats = dataset.summarize(train_ds, max_samples=5_000)
print(json.dumps(stats, indent=2))

In [ ]:
n_show = min(len(train_ds), 5_000)
rows = [dataset._sizes_of(train_ds, i) for i in range(n_show)]
n_res = np.array([r[0] for r in rows]); n_lig = np.array([r[1] for r in rows])
n_poc = np.array([len(train_ds.get_sample(i)["poc_pos"]) for i in range(n_show)])

fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].hist(n_lig, bins=range(0, CFG.max_ligand_atoms + 2), color="#3b82f6")
ax[0].set_title("ligand heavy atoms"); ax[0].set_xlabel("atoms")
ax[1].hist(n_poc, bins=40, color="#10b981")
ax[1].axvline(CFG.max_pocket_atoms, color="crimson", ls="--", label="cap")
ax[1].set_title("pocket atoms"); ax[1].legend()
ax[2].hist(n_res, bins=40, color="#f59e0b"); ax[2].set_title("pocket residues")
frac = pd.Series(stats["element_fraction"])
ax[3].bar(frac.index, frac.values, color="#8b5cf6"); ax[3].set_title("ligand element fraction")
ax[3].set_yscale("log")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"ligands at the {CFG.max_ligand_atoms}-atom cap: {(n_lig >= CFG.max_ligand_atoms).mean():.2%}")
print(f"pockets at the {CFG.max_pocket_atoms}-atom cap: {(n_poc >= CFG.max_pocket_atoms).mean():.2%}")

In [ ]:
# one complex end to end: the shapes the models will see
d = train_ds[0]
print(d)
print(f"\n{d.complex_id}  ligand={d.lig_name}")
print(f"  ligand atoms      {d.lig_pos.shape[0]:>5}   bonds (directed) {d.lig_bond_index.shape[1]:>6}")
print(f"  pocket atoms      {d.poc_pos.shape[0]:>5}   contacts         {d.poc_edge_index.shape[1]:>6}")
print(f"  cross edges       {d.inter_edge_index.shape[1]:>5}")
print(f"  covalent fraction of pocket edges: {d.poc_edge_attr[:, 0].mean():.3f}")
print(f"  ligand elements: {[chem.LIGAND_ELEMENTS[t] for t in d.lig_type.tolist()]}")

## 8 · Optional · PDBBind shards with affinity labels

The generator trains on BioLiP; the **ranker needs labels**, which only PDBBind/PDBBind+
provides. Download `refined-set` (or the general set) plus the `INDEX_*` files from
<https://www.pdbbind-plus.org.cn/download>, attach it as a Dataset, and run this section.

The CASF-2016 core set is forced into the test split: it is the standard held-out benchmark
and is a subset of the refined set, so leaving it in train would make the reported
correlation meaningless.

In [ ]:
AFFINITY_DIR = WORK / "graphs_affinity"

try:
    pdbbind = sources.PDBBindSource()
    print("set dir   :", pdbbind.set_dir)
    print("index file:", pdbbind.index_file)
    print(f"{len(pdbbind.labels):,} labelled entries")

    aff_specs = pdbbind.specs(require_affinity=True, prefer_pocket_file=True)
    print(f"{len(aff_specs):,} complexes with both files and a label")

    # CASF-2016 core set, if its index is present alongside
    core = sources.find_file(r"INDEX_core_data\.2016")
    holdout = set(sources.parse_pdbbind_index(core)) if core else None
    print("CASF-2016 core entries forced into test:", len(holdout or []))

    aff_split = sources.split_by_receptor(aff_specs, (0.8, 0.1, 0.1), seed=0, holdout_ids=holdout)
    AFFINITY_DIR.mkdir(parents=True, exist_ok=True)
    aff_manifests = {}
    for name in ("val", "test", "train"):
        aff_manifests[name] = dataset.build_shards(
            aff_split[name], AFFINITY_DIR, split=name, cfg=CFG,
            num_workers=max(1, os.cpu_count() or 2), chunk_size=64,
        )
    aff_train = dataset.ShardedComplexDataset(AFFINITY_DIR, "train")
    print("\naffinity label stats:", json.dumps(dataset.summarize(aff_train).get("affinity", {}), indent=2))
    HAS_AFFINITY_DATA = True
except FileNotFoundError as exc:
    print("PDBBind not found -> skipping affinity shards.")
    print(" ", exc)
    print("\nNotebook 03 needs these. Attach PDBBind and re-run this cell.")
    HAS_AFFINITY_DATA = False

## 9 · Ship the library with the output

Copying `src/molgen` into the output means notebooks 02–04 need **one** input dataset: this
notebook's output. Save the version, click **Save Version → Save & Run All**, then add the
output as a Dataset input to the next notebook.

In [ ]:
import shutil

dst = WORK / "src" / "molgen"
if Path(MOLGEN_SRC, "molgen").resolve() != dst.resolve():
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(Path(MOLGEN_SRC) / "molgen", dst,
                    ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
print("library copied to:", dst)

print("\noutput tree:")
for p in sorted(WORK.rglob("*")):
    if p.is_file() and "__pycache__" not in str(p):
        rel = p.relative_to(WORK)
        print(f"  {str(rel):<48} {p.stat().st_size / 1e6:>9.2f} MB")
print(f"\ntotal: {sum(f.stat().st_size for f in WORK.rglob('*') if f.is_file()) / 1e9:.2f} GB")

---
**Next:** `02_train_pocket_conditioned_diffusion.ipynb` — attach this output and train the generator.